# Claude Code CLI — Notebook interactivo

Este notebook simula los patrones de Claude Code usando la API de Anthropic directamente,
para que puedas entender qué ocurre bajo el capó cuando usas el CLI.

**Claude Code en la práctica = API de Anthropic + acceso al filesystem + bucle de herramientas**

In [ ]:
import anthropic
import os
import json
from pathlib import Path

client = anthropic.Anthropic()
print("Cliente Anthropic listo")

## 1. Lo que Claude Code hace al arrancar

Al lanzar `claude` en un directorio, Claude Code:
1. Lee el CLAUDE.md (si existe)
2. Escanea la estructura del proyecto
3. Carga ese contexto en el system prompt

In [ ]:
def leer_claude_md(directorio: str = ".") -> str:
    """Simula la carga del CLAUDE.md que hace Claude Code al arrancar."""
    ruta = Path(directorio) / "CLAUDE.md"
    if ruta.exists():
        return ruta.read_text()
    return "(No existe CLAUDE.md en este proyecto)"

def escanear_estructura(directorio: str = ".", max_archivos: int = 30) -> str:
    """Genera un árbol de estructura del proyecto."""
    lineas = []
    base = Path(directorio)
    ignorar = {'.git', '__pycache__', 'node_modules', '.venv', 'venv', '.next', 'dist'}
    
    def recorrer(path: Path, prefijo: str = "", nivel: int = 0):
        if nivel > 3 or len(lineas) > max_archivos:
            return
        try:
            items = sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name))
            for item in items:
                if item.name.startswith('.') or item.name in ignorar:
                    continue
                lineas.append(f"{prefijo}{'📁 ' if item.is_dir() else '📄 '}{item.name}")
                if item.is_dir():
                    recorrer(item, prefijo + "  ", nivel + 1)
        except PermissionError:
            pass
    
    recorrer(base)
    return "\n".join(lineas[:max_archivos])

# Escanear el directorio actual del notebook
cwd = Path(".").resolve()
print(f"Directorio: {cwd}")
print(f"\nEstructura detectada:")
print(escanear_estructura())

In [ ]:
# Simular el system prompt inicial de Claude Code
def construir_system_prompt(directorio: str = ".") -> str:
    claude_md = leer_claude_md(directorio)
    estructura = escanear_estructura(directorio)
    
    return f"""Eres Claude Code, un asistente de programación con acceso completo al proyecto.

INSTRUCCIONES DEL PROYECTO (CLAUDE.md):
{claude_md}

ESTRUCTURA DEL PROYECTO:
{estructura}

Puedes leer archivos, editar código, ejecutar comandos y ayudar con cualquier tarea de desarrollo.
Sé concreto, preciso y sigue las convenciones del proyecto."""

system = construir_system_prompt()
print(f"System prompt generado ({len(system)} chars)")
print(system[:500] + "...")

## 2. Modo interactivo simulado

Simulamos el REPL de Claude Code con memoria de conversación.

In [ ]:
class ClaudeCodeSimulado:
    """Simula el REPL de Claude Code."""
    
    def __init__(self, directorio: str = "."):
        self.historial: list[dict] = []
        self.system = construir_system_prompt(directorio)
        self.tokens_totales = {"input": 0, "output": 0}
    
    def preguntar(self, mensaje: str, modelo: str = "claude-sonnet-4-6") -> str:
        self.historial.append({"role": "user", "content": mensaje})
        
        resp = client.messages.create(
            model=modelo,
            max_tokens=1000,
            system=self.system,
            messages=self.historial
        )
        
        respuesta = resp.content[0].text
        self.historial.append({"role": "assistant", "content": respuesta})
        self.tokens_totales["input"] += resp.usage.input_tokens
        self.tokens_totales["output"] += resp.usage.output_tokens
        
        return respuesta
    
    def compact(self):
        """Equivalente a /compact — comprime el historial."""
        if len(self.historial) < 4:
            return
        resumen_resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            messages=[{"role": "user", "content": 
                f"Resume esta conversación en 3-5 puntos clave:\n{json.dumps(self.historial[:-2])}"}]
        )
        resumen = resumen_resp.content[0].text
        self.historial = [
            {"role": "user", "content": f"Resumen de conversación anterior:\n{resumen}"},
            {"role": "assistant", "content": "Entendido, continúo con ese contexto."}
        ] + self.historial[-2:]
        print(f"[/compact] Historial comprimido a {len(self.historial)} mensajes")
    
    def cost(self) -> str:
        """Equivalente a /cost."""
        coste = (self.tokens_totales["input"] * 3.00 + 
                 self.tokens_totales["output"] * 15.00) / 1_000_000
        return f"Tokens: {self.tokens_totales['input']} in / {self.tokens_totales['output']} out | Coste: ${coste:.4f}"
    
    def clear(self):
        """Equivalente a /clear."""
        self.historial = []
        print("[/clear] Contexto limpiado")

# Iniciar sesión simulada
cc = ClaudeCodeSimulado()
print("Claude Code simulado iniciado")

In [ ]:
# Sesión de exploración de codebase (equivalente a un REPL real)
resp = cc.preguntar(
    "Eres un asistente de desarrollo. Tengo un proyecto Python con FastAPI. "
    "Explícame qué convenciones debería seguir para los nombres de variables "
    "y funciones en Python moderno (3.11+)."
)
print(resp)

In [ ]:
# Segunda pregunta en la misma sesión (recuerda el contexto)
resp2 = cc.preguntar(
    "Y para los modelos Pydantic que uso como schemas de la API, "
    "¿qué convenciones específicas recomiendas?"
)
print(resp2)
print(f"\n[/cost] {cc.cost()}")

## 3. Generar un CLAUDE.md automáticamente (/init)

In [ ]:
def generar_claude_md(directorio: str = ".") -> str:
    """Equivalente al comando /init de Claude Code."""
    estructura = escanear_estructura(directorio)
    
    # Leer archivos de configuración que existan
    configs = []
    for nombre in ["pyproject.toml", "package.json", "requirements.txt", "Dockerfile", ".env.example"]:
        ruta = Path(directorio) / nombre
        if ruta.exists():
            configs.append(f"=== {nombre} ===\n{ruta.read_text()[:500]}")
    
    contexto_config = "\n".join(configs) if configs else "No se encontraron archivos de configuración"
    
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        messages=[{"role": "user", "content": f"""Genera un CLAUDE.md completo y útil para este proyecto.
Incluye: stack detectado, comandos clave, convenciones de código, reglas importantes.
Usa formato Markdown. Sé concreto y práctico.

ESTRUCTURA DEL PROYECTO:
{estructura}

ARCHIVOS DE CONFIGURACIÓN:
{contexto_config}"""}]
    )
    return resp.content[0].text

print("Generando CLAUDE.md...")
claude_md_generado = generar_claude_md()
print(claude_md_generado)

## 4. Modo headless: uso en scripts y CI

Equivalente a `claude --print "pregunta"` desde la línea de comandos.

In [ ]:
def claude_headless(prompt: str, modelo: str = "claude-haiku-4-5-20251001") -> str:
    """Equivalente a `claude --print 'prompt'` — respuesta única sin REPL."""
    resp = client.messages.create(
        model=modelo,
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text

# Ejemplos de uso headless (típicos en CI/CD)
usos_headless = [
    ("Genera un mensaje de commit para: añadir endpoint POST /api/invoices con extracción de datos via Claude",
     "🔖 Commit message:"),
    ("¿Este código tiene algún problema de seguridad potencial?\n```python\nquery = f'SELECT * FROM users WHERE email = {email}'\n```",
     "🔒 Security review:"),
    ("Genera 3 casos de test para una función que valida un IBAN español",
     "🧪 Test cases:"),
]

for prompt, etiqueta in usos_headless:
    print(f"\n{etiqueta}")
    print("=" * 50)
    print(claude_headless(prompt))

## Resumen: Claude Code CLI = estos tres patrones

| Patrón | Código equivalente | Cuándo usar |
|---|---|---|
| REPL con memoria | `ClaudeCodeSimulado.preguntar()` | Exploración y desarrollo iterativo |
| Headless | `claude_headless()` | Scripts, CI/CD, automatización |
| /init | `generar_claude_md()` | Al empezar en un proyecto nuevo |
| /compact | `cc.compact()` | Sesiones largas para ahorrar tokens |
| /cost | `cc.cost()` | Monitorizar el gasto de la sesión |